# HellanodikAI — Fine-Tuning Pipeline (Kaggle Dual T4 GPU)

Notebook ini dirancang untuk dijalankan di Kaggle Notebook dengan GPU accelerator (NVIDIA T4 x2 atau Single T4). Notebook ini melakukan:
1. Setup dependensi (Unsloth, TRL, PEFT, Hugging Face).
2. Generasi SFT dataset sintetis berbasis `pasal_corpus.json` menggunakan model `Qwen2.5-7B-Instruct` di GPU.
3. Fine-tuning model base `GoToCompany/llama3-8b-cpt-sahabatai-v1-instruct` menggunakan QLoRA via Unsloth.
4. Menyimpan dan melakukan upload hasil fine-tune (Adapter + GGUF) ke Hugging Face Hub.

## 1. Setup Dependensi & Environment

In [ ]:
# Install Unsloth & dependensi PyTorch
!pip install --no-deps "xformers<0.0.26" "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft transformers accelerate bitsandbytes loguru PyMuPDF rank-bm25 faiss-cpu sentence-transformers

import os
import torch
from kaggle_secrets import UserSecretsClient

# Ambil token Hugging Face dari Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("Token Hugging Face berhasil dimuat dari Secrets.")
except Exception as e:
    print("Warning: Gagal memuat token HF. Pastikan Anda telah memasukkan secret 'HF_TOKEN' di menu Add-ons -> Secrets di Kaggle.", e)

## 2. Generate SFT Dataset secara Otomatis
Kita akan menggunakan script generator yang telah dibuat untuk membuat pasangan tanya-jawab hukum menggunakan GPU Kaggle secara efisien.

In [ ]:
# Jalankan dataset generator sintetis
# Script ini akan men-download Qwen2.5-7B-Instruct dan menggunakannya untuk generate QA pairs dari pasal_corpus.json
print("=== Menjalankan Generasi Dataset SFT ===")
!python src/data_pipeline/generate_sft_dataset.py

## 3. Load Base Model dengan Unsloth (4-bit)
Kita memuat model `GoToCompany/llama3-8b-cpt-sahabatai-v1-instruct` menggunakan Unsloth agar proses training berjalan 2-3x lebih cepat dan menghemat VRAM (hanya memakan ~7GB VRAM).

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Konteks window
dtype = None # Auto-detection (float16 untuk T4 GPU)
load_in_4bit = True # 4-bit quantization untuk menghemat VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "GoToCompany/llama3-8b-cpt-sahabatai-v1-instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## 4. Setup Parameter QLoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Sangat hemat memori
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("Model PEFT (QLoRA) berhasil di-setup.")

## 5. Load Dataset dan Formatting Chat Template

In [ ]:
from datasets import load_dataset

# Load dataset hasil generator tadi
dataset = load_dataset("json", data_files="data/processed/sft_dataset.jsonl", split="train")

# Format ke template chat Llama-3/Sahabat-AI
def format_prompts(examples):
    instructions = examples["instruction"]
    responses = examples["response"]
    contexts = examples["context"]
    
    texts = []
    for inst, resp, ctx in zip(instructions, responses, contexts):
        # Gabungkan konteks RAG ke dalam prompt instruksi
        user_prompt = f"Konteks Hukum Rujukan:\n{ctx}\n\nPertanyaan:\n{inst}"
        
        # Llama-3 Chat Template
        text = (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
            f"Anda adalah HellanodikAI, asisten edukasi hukum pidana Indonesia. Jawablah pertanyaan dengan jujur, ramah, objektif, dan sertakan rujukan pasal secara akurat.<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n{user_prompt}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n{resp}<|eot_id|>"
        )
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(format_prompts, batched = True)
print(f"Dataset berhasil diformat. Jumlah sampel training: {len(dataset)}")

## 6. Training Model (Fine-Tuning)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Bisa True jika ingin memadatkan context window
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 60, # Jumlah step training (sesuaikan jika dataset lebih banyak)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("=== Memulai Training Model ===")
trainer_stats = trainer.train()

## 7. Model Saving & Hugging Face Upload
Kita menyimpan adapter hasil training ke disk dan langsung meng-upload-nya ke Hugging Face Hub (repo name harus ber-prefix `Llama-` sesuai lisensi model).

In [ ]:
repo_name = "Llama-HellanodikAI-8B"

# 1. Simpan LoRA adapters secara lokal
model.save_pretrained("hellanodikai_lora_model")
tokenizer.save_pretrained("hellanodikai_lora_model")

# 2. Upload LoRA adapters ke Hugging Face Hub
try:
    print(f"Melakukan upload LoRA adapters ke HF Hub: {repo_name}...")
    model.push_to_hub(repo_name, use_auth_token=True)
    tokenizer.push_to_hub(repo_name, use_auth_token=True)
    print("Upload adapter berhasil!")
except Exception as e:
    print("Gagal mengupload adapter ke HF. Periksa token Anda.", e)

## 8. Export ke GGUF (Untuk CPU Lokal) & Push ke HF Hub
Unsloth memiliki fitur luar biasa untuk langsung melakukan merge LoRA ke base model dan mengekspornya ke format GGUF (`Q4_K_M`) secara otomatis, lalu melakukan push ke Hugging Face Hub.

In [ ]:
# Export ke Q4_K_M GGUF format dan langsung push ke Hugging Face Hub
try:
    print("Melakukan merge model dan konversi ke GGUF Q4_K_M...")
    # Prosedur merge + GGUF export + push ke HF Hub
    model.push_to_hub_gguf(
        repo_id = repo_name,
        tokenizer = tokenizer,
        quantization_method = "q4_k_m", # Quantization yang optimal untuk CPU
        use_auth_token = True
    )
    print("=== PROSES MERGE DAN UPLOAD GGUF SELESAI DENGAN SUKSES ===")
except Exception as e:
    print("Gagal melakukan konversi GGUF / Push ke HF. Periksa dependensi gguf-py atau memori disk.", e)